In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:

from openai import OpenAI
openai_client = OpenAI()

In [4]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [5]:
llm('Hey whats up')

'Not much—just here and ready to help. What’s up with you?'

In [12]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely — if the course is currently open, you can usually join now.

A couple of things to check:
- **Enrollment deadline:** Some courses allow late enrollment; others don’t.
- **Start date / access:** You may be able to join immediately, but if the course has already started, you might need to catch up on missed material.
- **Prerequisites or approval:** Some courses require instructor approval or have a waitlist.

If you want, I can help you draft a quick message to the course organizer asking whether late enrollment is still possible.


In [13]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [14]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still open.


In [33]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [6]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [7]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1368

In [8]:
documents[30]

{'id': 'a8219681ec',
 'course': 'data-engineering-zoomcamp',
 'section': 'Environment & Setup',
 'question': 'Why does the course use GCP and not AWS or Azure?',
 'answer': 'For uniformity across the cohort. The course uses BigQuery, which is GCP-only, and most students already have a Google account that works for sign-up. The concepts (data warehouse, object storage, IaC) translate to AWS/Azure, but the lessons are recorded against GCP.\n\nYou can use a different cloud for your project — see [the AWS/Snowflake/Azure FAQ](#19aaf65b2a) for the tradeoffs.'}

In [26]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [30]:
search_results = index.search(
    question, 
    boost_dict={'question': 2.0, 'section':0.5, 'answer':0.1},
    filter_dict={'course':"llm-zoomcamp"}, 
    num_results=5
    )

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict={'question': 2.0, 'section':0.5}
    filter_dict={'course':course}
    
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict= filter_dict, 
        num_results=5
    )   

In [42]:
search_results = search(question)

In [44]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [45]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [46]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [66]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [67]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [61]:
prompt = build_prompt(question, search_results)

In [62]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [68]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [ ]:
response.output_text

'Yes — you can join now.\n\nYou can start anytime, but if you want a certificate, you’ll need to submit your project while submissions are still open and be part of the live cohort.'

In [70]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_08cb9aff14c5afea006a493479985c819c812b585a447aee92",
  "created_at": 1783182457.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_08cb9aff14c5afea006a49347a2904819ca20362afa81031d3",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can join now.\n\nYou can start anytime, but if you want a certificate, you’ll need to submit your project while submissions are still open and be part of the live cohort.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1783182458.0,
  "conversation": null,
  "max_outpu

In [72]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [73]:
response.output_text

'Yes, you can still join now. If you want a certificate, make sure you submit your project while submissions are still open.'

In [74]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [75]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [77]:
answer = rag(question)
print(answer)

Yes, you can still join now. If you want a certificate, make sure to submit your project while the course is still accepting submissions.
